# Cincinnati Relief-Adjusted LVT Workflow

This notebook models a Cincinnati city-only LVT scenario using the local ArcGIS parcel extract plus the Hamilton County exempt / abatement export.

Key modeling rules in this notebook:
- Use only parcels in `CINTI CORP-CINTI CSD`
- Ignore TIF values
- Remove fully exempt parcels where `abatement_value + exempt_value >= market total`
- Apply relief to improvement value first, then any leftover to land value
- Apply the Ohio `35%` assessment ratio after relief allocation
- Run current-tax and split-rate modeling on the adjusted assessed base


In [ ]:
import json
import os
import sys
import tempfile
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import requests
import seaborn as sns

sys.path.insert(0, str(Path('../..').resolve()))
REPO_ROOT = Path('../..').resolve()

from lvt.census_utils import get_census_data
from lvt.lvt_utils import calculate_category_tax_summary, calculate_current_tax, model_split_rate_tax
from lvt.viz import (
    create_quintile_summary,
    create_spokane_property_category_chart,
    create_threshold_change_chart,
    plot_upside_down_quintile_bars,
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
pd.options.display.float_format = '{:,.2f}'.format


In [ ]:
DATA_DIR = REPO_ROOT / 'examples' / 'data' / 'cincinnati'
DATA_DIR.mkdir(parents=True, exist_ok=True)

PARCEL_PATH = DATA_DIR / 'cincinnati_20260119.gpq'
RELIEF_PATH = DATA_DIR / 'ExemptAbatedTif2025.xlsx'
RELIEF_URL = 'https://www.hamiltoncountyauditor.org/download/TIFAbateExempt/ExemptAbatedTif2025.xlsx'
OH_BG_ZIP_URL = 'https://www2.census.gov/geo/tiger/TIGER2022/BG/tl_2022_39_bg.zip'

CITY_TAX_DISTRICT = 'CINTI CORP-CINTI CSD'
CITY_MILLAGE = 6.1
ASSESSMENT_RATIO = 0.35
LAND_IMPROVEMENT_RATIO = 4.0
HAMILTON_COUNTY_FIPS = '39061'
ACS_YEAR = 2022
HOLD_EVEN_CATEGORIES = {'Agricultural', 'Parks / Recreation'}

CATEGORY_MAP = {
    'SF': 'Single Family Residential',
    'TF': 'Small Multi-Family (2-4 units)',
    'MF': 'Large Multi-Family (5+ units)',
    'VA': 'Vacant Land',
    'C': 'Retail/Service/Commercial',
    'O': 'Office / Other Commercial',
    'MU': 'Mixed Use',
    'LI': 'Light Industrial',
    'IN': 'Industrial',
    'HI': 'Hospitality / High Intensity Commercial',
    'PS': 'Public / Semi-Public',
    'PU': 'Public Utility / Railroad',
    'ED': 'Education / Institutional',
    'PR': 'Parks / Recreation',
    'CH': 'Church / Religious',
    'AG': 'Agricultural',
    'MH': 'Mobile Home',
    'NA': 'Other / NA',
    '?': 'Other / NA',
}


def ensure_relief_file():
    if RELIEF_PATH.exists():
        return
    response = requests.get(RELIEF_URL, timeout=120)
    response.raise_for_status()
    RELIEF_PATH.write_bytes(response.content)


def normalize_digits(series):
    return series.astype(str).str.replace(r'\D', '', regex=True)


## Step 1: Load Parcel And Relief Data

In [ ]:
ensure_relief_file()

parcels = gpd.read_parquet(PARCEL_PATH)
parcels = parcels[parcels['TAXDST_DIS'].astype(str).eq(CITY_TAX_DISTRICT)].copy()

for col in ['MKTLND', 'MKTIMP', 'MKT_TOTAL_VAL', 'ANNUAL_TAXES']:
    parcels[col] = pd.to_numeric(parcels[col], errors='coerce').fillna(0.0)

parcels['parcel_join_id'] = normalize_digits(parcels['AUDPTYID'])
parcels['PROPERTY_CATEGORY'] = parcels['EXLUCODE'].astype(str).map(CATEGORY_MAP).fillna('Other / NA')

relief = pd.read_excel(
    RELIEF_PATH,
    usecols=['parcel_number_unformatted', 'exempt_value', 'abatement_value'],
).copy()
relief['parcel_join_id'] = normalize_digits(relief['parcel_number_unformatted'])
relief['exempt_value'] = pd.to_numeric(relief['exempt_value'], errors='coerce').fillna(0.0)
relief['abatement_value'] = pd.to_numeric(relief['abatement_value'], errors='coerce').fillna(0.0)
relief = relief.groupby('parcel_join_id', as_index=False).agg(
    exempt_value=('exempt_value', 'sum'),
    abatement_value=('abatement_value', 'sum'),
)
relief['total_relief'] = relief['exempt_value'] + relief['abatement_value']

parcels = parcels.merge(relief, on='parcel_join_id', how='left')
for col in ['exempt_value', 'abatement_value', 'total_relief']:
    parcels[col] = parcels[col].fillna(0.0)

pd.DataFrame({
    'city_parcels_before_relief_filter': [len(parcels)],
    'joined_relief_rows': [int((parcels['total_relief'] > 0).sum())],
    'total_exempt_value': [parcels['exempt_value'].sum()],
    'total_abatement_value': [parcels['abatement_value'].sum()],
    'total_relief': [parcels['total_relief'].sum()],
})

## Step 2: Remove Fully Exempt Parcels And Apply Relief

Relief is applied to improvement value first, then to land value. Fully exempt parcels are removed before the rest of the analysis.


In [ ]:
parcels['full_exmp'] = (parcels['total_relief'] >= parcels['MKT_TOTAL_VAL']).astype(int)
full_exempt_removed = int(parcels['full_exmp'].sum())

remaining_relief = (parcels['total_relief'] - parcels['MKTIMP']).clip(lower=0.0)
parcels['taxable_market_improvement'] = (parcels['MKTIMP'] - parcels['total_relief']).clip(lower=0.0)
parcels['taxable_market_land'] = (parcels['MKTLND'] - remaining_relief).clip(lower=0.0)
parcels['taxable_market_total'] = parcels['taxable_market_land'] + parcels['taxable_market_improvement']

parcels_model = parcels[parcels['full_exmp'] == 0].copy()

pd.DataFrame({
    'full_exempt_removed': [full_exempt_removed],
    'modeled_parcels': [len(parcels_model)],
    'taxable_market_land_sum': [parcels_model['taxable_market_land'].sum()],
    'taxable_market_improvement_sum': [parcels_model['taxable_market_improvement'].sum()],
    'taxable_market_total_sum': [parcels_model['taxable_market_total'].sum()],
})

## Step 3: Apply Assessment Ratio And Recreate Current Revenue

In [ ]:
parcels_model['taxable_assessed_land'] = parcels_model['taxable_market_land'] * ASSESSMENT_RATIO
parcels_model['taxable_assessed_improvement'] = parcels_model['taxable_market_improvement'] * ASSESSMENT_RATIO
parcels_model['taxable_assessed_total'] = parcels_model['taxable_market_total'] * ASSESSMENT_RATIO
parcels_model['millage_rate'] = CITY_MILLAGE

current_revenue, _, modeled = calculate_current_tax(
    df=parcels_model,
    tax_value_col='taxable_assessed_total',
    millage_rate_col='millage_rate',
)

pd.DataFrame({
    'current_city_revenue': [current_revenue],
    'city_millage': [CITY_MILLAGE],
    'assessment_ratio': [ASSESSMENT_RATIO],
    'taxable_assessed_total_sum': [modeled['taxable_assessed_total'].sum()],
})

## Step 4: Run Revenue-Neutral 4:1 Split-Rate Scenario

In [ ]:
held_even_mask = modeled['PROPERTY_CATEGORY'].isin(HOLD_EVEN_CATEGORIES)
held_even_parcels = modeled[held_even_mask].copy()
split_rate_parcels = modeled[~held_even_mask].copy()
split_rate_target_revenue = split_rate_parcels['current_tax'].sum()

land_millage, improvement_millage, split_revenue, split_rate_modeled = model_split_rate_tax(
    df=split_rate_parcels,
    land_value_col='taxable_assessed_land',
    improvement_value_col='taxable_assessed_improvement',
    current_revenue=split_rate_target_revenue,
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

held_even_parcels['new_tax'] = held_even_parcels['current_tax']
held_even_parcels['tax_change'] = 0.0
held_even_parcels['tax_change_pct'] = 0.0

modeled = pd.concat([split_rate_modeled, held_even_parcels], ignore_index=False).sort_index()
modeled['tax_change'] = modeled['new_tax'] - modeled['current_tax']
modeled['tax_change_pct'] = np.where(
    modeled['current_tax'] > 0,
    modeled['tax_change'] / modeled['current_tax'] * 100,
    0.0,
)

pd.DataFrame({
    'land_millage': [land_millage],
    'improvement_millage': [improvement_millage],
    'current_revenue': [current_revenue],
    'split_revenue_non_held_even': [split_revenue],
    'held_even_revenue': [held_even_parcels['current_tax'].sum()],
    'split_revenue_total': [modeled['new_tax'].sum()],
    'held_even_parcels': [int(held_even_mask.sum())],
})

## Step 5: Property Category Summary

In [ ]:
category_summary = calculate_category_tax_summary(
    modeled,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
category_summary

In [ ]:
category_plot_df = category_summary[
    (category_summary['property_count'] >= 25)
    & (~category_summary['PROPERTY_CATEGORY'].isin(HOLD_EVEN_CATEGORIES))
].copy()

create_threshold_change_chart(
    category_plot_df,
    title='Cincinnati: Share of Parcels with Tax Changes Above 10% by Property Category',
    threshold=10,
    min_count=25,
    figsize=(12, 8),
)

create_spokane_property_category_chart(
    category_plot_df,
    title='Cincinnati: Median Tax Change by Property Category',
    min_count=25,
    figsize_width=18,
)


## Step 6: Attach Census Data For Progressivity Analysis

In [ ]:
census_data = get_census_data(HAMILTON_COUNTY_FIPS, year=ACS_YEAR)
with tempfile.TemporaryDirectory() as tmpdir:
    zip_path = Path(tmpdir) / 'oh_bg.zip'
    response = requests.get(OH_BG_ZIP_URL, timeout=120)
    response.raise_for_status()
    zip_path.write_bytes(response.content)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(tmpdir)
    block_groups = gpd.read_file(tmpdir)

block_groups = block_groups[block_groups['COUNTYFP'].eq(HAMILTON_COUNTY_FIPS[2:])].copy()
block_groups['std_geoid'] = block_groups['GEOID']
block_groups = block_groups.merge(
    census_data[['std_geoid', 'median_income', 'total_pop', 'minority_pct', 'black_pct']],
    on='std_geoid',
    how='left',
)

if str(modeled.crs) != str(block_groups.crs):
    block_groups = block_groups.to_crs(modeled.crs)

centroids = modeled.to_crs(3857).copy()
centroids.geometry = centroids.geometry.centroid
centroids = centroids.to_crs(modeled.crs)

parcel_with_census = gpd.sjoin(
    centroids,
    block_groups[['std_geoid', 'median_income', 'total_pop', 'minority_pct', 'black_pct', 'geometry']],
    how='left',
    predicate='within',
).drop(columns=['index_right'])
parcel_with_census.geometry = modeled.geometry.values

parcel_with_census[['std_geoid', 'median_income', 'minority_pct', 'black_pct']].head()

## Step 7: Income And Demographic Quintiles

In [ ]:
non_vacant_taxable = parcel_with_census[
    parcel_with_census['PROPERTY_CATEGORY'].ne('Vacant Land')
    & parcel_with_census['median_income'].notna()
    & (parcel_with_census['current_tax'] > 0)
].copy()

residential_categories = {
    'Single Family Residential',
    'Small Multi-Family (2-4 units)',
    'Large Multi-Family (5+ units)',
    'Mobile Home',
}

residential_taxable = non_vacant_taxable[
    non_vacant_taxable['PROPERTY_CATEGORY'].isin(residential_categories)
].copy()

income_quintiles = create_quintile_summary(residential_taxable, 'median_income', 'median_income')
minority_quintiles = create_quintile_summary(residential_taxable, 'minority_pct', 'minority_pct')
black_quintiles = create_quintile_summary(residential_taxable, 'black_pct', 'black_pct')

income_quintiles

In [ ]:
def plot_green_quintile_bars(summary_df, title, quintile_col, value_col='median_tax_change_pct', figsize=(13, 8)):
    plot_df = summary_df.copy()
    labels = plot_df[quintile_col].astype(str).tolist()
    vals = plot_df[value_col].astype(float).tolist()
    x = np.arange(len(labels))
    max_abs = max([abs(v) for v in vals] + [1.0])
    green_palette = sns.color_palette('Greens', n_colors=6)
    colors = []
    for val in vals:
        intensity = abs(val) / max_abs
        idx = min(5, max(1, int(round(intensity * 5))))
        colors.append(green_palette[idx])

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(x, vals, color=colors, edgecolor='black', width=0.7)
    ax.axhline(0, color='black', linewidth=1, linestyle='dotted')
    ax.grid(False)
    ax.yaxis.set_visible(False)
    ax.set_ylabel('')
    ax.set_xlabel('')
    ax.set_title(title, weight='bold', pad=30)
    sns.despine(left=True, right=True, top=True, bottom=True)

    for bar, val in zip(bars, vals):
        ypos = 0.5 if abs(val) < 0.01 else val / 2
        ax.annotate(
            f'{val:.1f}%',
            xy=(bar.get_x() + bar.get_width() / 2, ypos),
            xytext=(0, 0),
            textcoords='offset points',
            ha='center',
            va='center' if abs(val) >= 0.01 else 'bottom',
            fontsize=13,
            color='black',
            fontweight='bold',
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontweight='bold')
    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')
    ymin = min(vals + [0])
    ymax = max(vals + [0])
    ypad = max(abs(ymin), abs(ymax)) * 0.15 if max(abs(ymin), abs(ymax)) > 0 else 1.0
    ax.set_ylim(ymin - ypad, ymax + ypad)
    plt.tight_layout()
    plt.show()

plot_green_quintile_bars(
    income_quintiles,
    title='Cincinnati: Median Tax Change by Neighborhood Income (All Residential)',
    quintile_col='median_income_quintile',
    value_col='median_tax_change_pct',
    figsize=(13, 8),
)

plot_green_quintile_bars(
    minority_quintiles,
    title='Cincinnati: Median Tax Change by Minority % Quintile (All Residential)',
    quintile_col='minority_pct_quintile',
    value_col='median_tax_change_pct',
    figsize=(13, 8),
)


In [ ]:
display(income_quintiles)
display(minority_quintiles)
display(black_quintiles)

In [ ]:
block_group_summary = (
    parcel_with_census[
        (parcel_with_census['median_income'] > 0)
        & (parcel_with_census['current_tax'] > 0)
    ]
    .groupby('std_geoid')
    .agg(
        median_income=('median_income', 'first'),
        minority_pct=('minority_pct', 'first'),
        black_pct=('black_pct', 'first'),
        total_current_tax=('current_tax', 'sum'),
        total_new_tax=('new_tax', 'sum'),
    )
    .reset_index()
)
block_group_summary['bg_tax_change_pct'] = (
    (block_group_summary['total_new_tax'] - block_group_summary['total_current_tax'])
    / block_group_summary['total_current_tax']
    * 100
)

pd.DataFrame({
    'income_correlation': [block_group_summary[['median_income', 'bg_tax_change_pct']].corr().iloc[0, 1]],
    'minority_correlation': [block_group_summary[['minority_pct', 'bg_tax_change_pct']].corr().iloc[0, 1]],
    'black_correlation': [block_group_summary[['black_pct', 'bg_tax_change_pct']].corr().iloc[0, 1]],
})

## Step 8: Export A Compact JSON Summary

In [ ]:
summary_payload = {
    'parcel_count_modeled': int(len(modeled)),
    'full_exempt_removed': int(full_exempt_removed),
    'current_revenue': float(current_revenue),
    'land_millage': float(land_millage),
    'improvement_millage': float(improvement_millage),
    'split_revenue': float(split_revenue),
    'total_relief_modeled_parcels': float(modeled['total_relief'].sum()),
    'category_summary': category_summary.to_dict(orient='records'),
    'income_quintiles': income_quintiles.to_dict(orient='records'),
    'minority_quintiles': minority_quintiles.to_dict(orient='records'),
    'black_quintiles': black_quintiles.to_dict(orient='records'),
}
print(json.dumps(summary_payload))

In [ ]:
# Export standardized CSV — do not remove or move above Census join
import sys
sys.path.insert(0, '../..')
from lvt.lvt_utils import save_standard_export

out_df = save_standard_export(
    df=modeled,
    city='cincinnati',
    output_path='../../analysis/data/cincinnati.csv',
    model_type='split_rate:4.0',
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    exempt_flag_col='full_exmp',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)


In [ ]:
# Standard report: category impact, income quintile, distribution
import sys
sys.path.insert(0, '../..')
from lvt.viz import create_city_report

create_city_report(out_df, 'cincinnati', show=True)
